<a href="https://colab.research.google.com/github/abhjaco/LLM/blob/main/Agentic_AI_Simple_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧪 Lab: Agentic AI Foundations – Simple Rule-Based Agent

## Objective
Build a simple **Agent** using:
- Planner
- Executor
- Tools
- Memory
- Agent Loop (Observe → Think → Act → Reflect)

This lab is **beginner-friendly** and runnable end-to-end.



## Architecture Overview

**User Query**
→ Planner (decides steps)  
→ Executor (calls tools)  
→ Memory (stores events)  
→ Final Answer


In [ ]:

from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
import json
import os
import time


## 1️⃣ Agent State (Short-Term Memory)

In [ ]:

@dataclass
class AgentState:
    user_query: str
    plan: List[Dict[str, Any]] = field(default_factory=list)
    step_index: int = 0
    scratchpad: Dict[str, Any] = field(default_factory=dict)
    observations: List[str] = field(default_factory=list)
    final_answer: Optional[str] = None


## 2️⃣ Persistent Memory (JSON Store)

In [ ]:

class JsonMemoryStore:
    def __init__(self, path: str = "agent_memory.json"):
        self.path = path
        self.data = self._load()

    def _load(self):
        if os.path.exists(self.path):
            with open(self.path, "r") as f:
                return json.load(f)
        return {"events": []}

    def save(self):
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def remember_event(self, event):
        self.data["events"].append(event)
        self.save()


## 3️⃣ Tools (Mock APIs)

In [ ]:

def tool_fetch_leave_balance(emp_id: str):
    return {"emp_id": emp_id, "leave_balance_days": 12}

def tool_fetch_salary(emp_id: str, month: str):
    return {"emp_id": emp_id, "month": month, "net_salary": 85650, "currency": "INR"}

def tool_categorize_expense(description: str, amount: float):
    if "uber" in description.lower():
        category = "Transport"
    elif "zomato" in description.lower():
        category = "Food"
    else:
        category = "Other"
    return {"description": description, "amount": amount, "category": category}


## 4️⃣ Planner (Rule-Based)

In [ ]:

def planner(user_query: str):
    q = user_query.lower()
    if "leave" in q:
        return [
            {"tool": "fetch_leave_balance", "args": {"emp_id": extract_emp_id(user_query)}},
            {"tool": "compose_answer", "args": {"kind": "leave"}}
        ]
    if "salary" in q:
        return [
            {"tool": "fetch_salary", "args": {"emp_id": extract_emp_id(user_query), "month": "Dec 2025"}},
            {"tool": "compose_answer", "args": {"kind": "salary"}}
        ]
    if "expense" in q:
        desc, amt = extract_expense(user_query)
        return [
            {"tool": "categorize_expense", "args": {"description": desc, "amount": amt}},
            {"tool": "compose_answer", "args": {"kind": "expense"}}
        ]
    return [{"tool": "compose_answer", "args": {"kind": "unknown"}}]


## 5️⃣ Executor (Runs One Step)

In [ ]:

def executor_step(state: AgentState, memory: JsonMemoryStore):
    if state.final_answer:
        return state

    step = state.plan[state.step_index]
    tool = step["tool"]
    args = step.get("args", {})

    if tool == "fetch_leave_balance":
        state.scratchpad["leave"] = tool_fetch_leave_balance(**args)

    elif tool == "fetch_salary":
        state.scratchpad["salary"] = tool_fetch_salary(**args)

    elif tool == "categorize_expense":
        state.scratchpad["expense"] = tool_categorize_expense(**args)

    elif tool == "compose_answer":
        state.final_answer = compose_answer(state, args["kind"])

    memory.remember_event({"tool": tool, "args": args})
    state.step_index += 1
    return state


## 6️⃣ Agent Loop

In [ ]:

def run_agent(user_query: str):
    state = AgentState(user_query=user_query)
    memory = JsonMemoryStore()
    state.plan = planner(user_query)

    while not state.final_answer:
        state = executor_step(state, memory)

    return state.final_answer


## 7️⃣ Helper Functions

In [ ]:

def extract_emp_id(text: str):
    return ''.join(c for c in text if c.isdigit()) or "UNKNOWN"

def extract_expense(text: str):
    parts = text.split()
    amt = float(parts[-1]) if parts[-1].isdigit() else 0
    desc = " ".join(parts[:-1])
    return desc, amt

def compose_answer(state: AgentState, kind: str):
    if kind == "leave":
        return f"Leave Balance: {state.scratchpad['leave']['leave_balance_days']} days"
    if kind == "salary":
        s = state.scratchpad['salary']
        return f"Salary for {s['month']}: {s['net_salary']} {s['currency']}"
    if kind == "expense":
        e = state.scratchpad['expense']
        return f"Expense categorized as {e['category']}"
    return "Sorry, I can't help with that."


## 8️⃣ Try It Out

In [ ]:

print(run_agent("Show my leave balance emp 123"))
print(run_agent("Show my salary emp 456"))
print(run_agent("Categorize expense Uber 450"))
